# 06 — Regulators (TF–TF network and master regulators per state)

Cancer-agnostic port of GBM `06_regulators.ipynb`. **Only the config cell below needs editing for a new cancer.**
Needs `filteredCausalResults.csv` (causal filter at the end of notebook 02) and the MINER outputs in `<cancer>_output/`.

Causal flows are first limited to published driver genes of this cancer (`driver_tiers.py`, tiers known / candidate; cached in `{prefix}_data/driver_genes.csv`), then TFs that are barely expressed are dropped. Extra outputs: `driverCausalResults.csv` (flows kept, with `Tier`), `tf_expression_filter.csv`.

**Master regulators (paper method).** For each state: the causal TFs of the regulons active in that state; a LASSO per TF on the state TFs that have a binding site in its promoter; pruning with the one-standard-error rule; non-zero coefficients as edges; TFs ranked by outdegree/(indegree + 1), top `N_MASTER` kept. Per-state networks: `tf_tf_network_per_state.csv`; per-state counts: `master_regulator_state_summary.csv`. The global network above (`tf_tf_network.csv`, `tf_network_degrees.csv`) is kept only for the optional out/in column in 07.

In [1]:
# Necessary imports
import os
import pickle
import time
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import miner.miner as miner
from miner.miner import *

# CANCER CONFIGURATION: EDIT TO GET TF-TF NETWORK AND MASTER REGULATORS FOR CANCER OF CHOICE 
CANCER     = "HNSC"

# folder holding cancer run of MINER
base_dir   = r"causal_inf_new"

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's expression / mutation files (written by 01 and 02)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")

# corresponding MINER outputs (regulons, eigengenes, ...)
output_dir = os.path.join(cancer_dir, f"{CANCER.lower()}_output")

# z-scored expression data, gene symbols x patients
expr_path  = os.path.join(data_dir, f"{CANCER.lower()}_zscored.csv")

# TF expression filter: a TF must reach MIN_TPM in at least MIN_FRAC of tumours to be in the TF-TF network
MIN_TPM  = 1.0
MIN_FRAC = 0.20
log2_path = os.path.join(data_dir, f"{CANCER.lower()}_log2tpm.csv")

# Driver gene tiers: published driver calls per cancer (module driver_tiers.py in base_dir, shared by 02, 06 and 07)
import sys
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)
from driver_tiers import get_driver_tiers

assert os.path.isdir(output_dir), f"missing output folder: {output_dir}"
assert os.path.exists(expr_path), f"missing expression file: {expr_path}"
miner_dir = os.path.dirname(miner.__file__)
print(f"Cancer: {CANCER} | outputs: {output_dir}")

Cancer: HNSC | outputs: causal_inf_new\hnsc\hnsc_output


In [2]:
# causal_results: filtered somatic mutation causal flows (mutation, TF, regulon)
# Only the three columns this notebook uses are read, as categories: in heavily mutated cancers this file has millions of
# rows (UCEC: 2.7 million, 400 MB) and loading all 13 columns can exhaust memory and kill the kernel
causal_results = pd.read_csv(os.path.join(output_dir, "filteredCausalResults.csv"),
                             usecols=["Mutation", "Regulator", "Regulon"], dtype="category")
causal_results['Regulon'] = causal_results['Regulon'].astype(str).str.replace('R-', '', regex=False)  # match eigengene index

# exp_data: z-scored expression, gene symbols x patients
exp_data = pd.read_csv(expr_path, index_col=0)

# eigengenes: PC1 score per regulon per patient
eigengenes = pd.read_csv(os.path.join(output_dir, "eigengenes.csv"), index_col=0)

# transcriptional_states: {state ID: [patient IDs]}
transcriptional_states = read_json(os.path.join(output_dir, "transcriptional_states.json"))

# Ensembl <-> gene symbol mappings
idmap = pd.read_csv(os.path.join(miner_dir, "data", "identifier_mappings.txt"), sep="\t")
gene_name_map     = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))
symbol_to_ensembl = {v: k for k, v in ensembl_to_symbol.items()}

# TFBSDB v1 + v2 (computational) + ChEA (experimental) binding site databases
nd = os.path.join(miner_dir, "data", "network_dictionaries")
tfbsdb_tf_to_genes  = pickle.load(open(os.path.join(nd, "tfbsdb_tf_to_genes.pkl"), 'rb'))
tfbsdb2_tf_to_genes = pickle.load(open(os.path.join(nd, "tfbsdb2_tf_to_genes.pkl"), 'rb'))
chea_seq            = pickle.load(open(os.path.join(nd, "chea_seq.pkl"), 'rb'))

print("causal_results:", causal_results.shape)
print("exp_data:", exp_data.shape)
print("eigengenes:", eigengenes.shape)
print("transcriptional_states:", len(transcriptional_states), "states")

causal_results: (7898, 3)
exp_data: (19755, 502)
eigengenes: (7609, 502)
transcriptional_states: 21 states


In [3]:
# Driver filter: keep only causal flows that start from a published driver gene of this cancer
# In heavily mutated cancers most frequently mutated genes are passengers (TTN, MUC16, CSMD3, ...) that track total
# mutation burden, so their "causal" flows are confounded. Same gene list as 02 and 07 (driver_tiers.py):
# both tiers need an IntOGen driver call for this cancer: "known" = also in the Cancer Gene Census, "candidate" = not in it
driver_table = get_driver_tiers(CANCER, data_dir)
tier_of = dict(zip(driver_table["Gene"], driver_table["Tier"]))

if len(driver_table) == 0:
    # back to plain text columns
    causal_results = causal_results.astype(str)
    causal_results["Tier"] = "unfiltered"
    print(f"No driver list for {CANCER}: all {causal_results['Mutation'].nunique()} mutated genes are kept")
else:
    n_flows, n_genes, n_tfs = len(causal_results), causal_results["Mutation"].nunique(), causal_results["Regulator"].nunique()
    # back to plain text columns now that the table is small
    causal_results = causal_results[causal_results["Mutation"].isin(tier_of.keys())].astype(str)
    causal_results["Tier"] = causal_results["Mutation"].map(tier_of)
    print(f"Causal flows: {n_flows} -> {len(causal_results)} | mutated genes: {n_genes} -> {causal_results['Mutation'].nunique()} | "
          f"TFs: {n_tfs} -> {causal_results['Regulator'].nunique()}")
    print("Flows per driver:")
    print(causal_results.groupby(["Tier", "Mutation"]).size().sort_values(ascending=False).to_string())

    # Listed drivers without flows: mutated in < 5% of patients, or mainly altered by copy number / fusion
    print("Listed drivers with no flows:", len(set(tier_of) - set(causal_results["Mutation"])), "of", len(tier_of))

# Saved so 06's input after the filter can be inspected
causal_results.to_csv(os.path.join(output_dir, "driverCausalResults.csv"), index=False)


Driver genes for HNSC loaded from cache: {'known': 68, 'candidate': 4}
IntOGen score >= 0.6: 19 of 72 driver calls kept: {'known': 16, 'candidate': 3}
Causal flows: 7898 -> 7898 | mutated genes: 13 -> 13 | TFs: 345 -> 345
Flows per driver:
Tier       Mutation
known      TP53        1568
           NSD1        1416
           CASP8       1283
           CDKN2A       990
           HRAS         912
           NOTCH1       695
           PIK3CA       235
candidate  AJUBA        235
known      FAT1         208
           EP300        135
           NFE2L2       115
           FBXW7         92
           KMT2D         14
Listed drivers with no flows: 6 of 19


In [4]:
# TFs implicated in causal flows
causal_tfs_symbols = causal_results['Regulator'].unique().tolist()

# Drop TFs that are barely expressed in this cancer
# Without this, TFs with near-zero noisy expression get LASSO edges and, having no regulators of their own, top the out/in ranking
# log2(TPM+1), gene symbols x patients (from 01); only the TF rows are kept in memory
log_tpm = pd.read_csv(log2_path, index_col=0)
tf_log_tpm = log_tpm.reindex(causal_tfs_symbols).dropna()
del log_tpm

# Fraction of tumours in which each TF reaches MIN_TPM
frac_expressed = (tf_log_tpm >= np.log2(MIN_TPM + 1)).mean(axis=1)
expressed_tfs  = frac_expressed[frac_expressed >= MIN_FRAC].index.tolist()
dropped_tfs    = sorted(set(causal_tfs_symbols) - set(expressed_tfs))

print(f"TFs kept: {len(expressed_tfs)} / {len(causal_tfs_symbols)} (>= {MIN_TPM} TPM in >= {MIN_FRAC:.0%} of tumours)")
print("Dropped:", dropped_tfs)

# Saved so the choice of threshold can be reviewed
frac_expressed.rename("frac_expressed").to_csv(os.path.join(output_dir, "tf_expression_filter.csv"))

causal_tfs_symbols = expressed_tfs
causal_tfs_ensembl = [t for t in (symbol_to_ensembl.get(tf) for tf in causal_tfs_symbols) if t is not None]
print(f"TFs from causal results: {len(causal_tfs_symbols)} | mapped to Ensembl: {len(causal_tfs_ensembl)}")

# TF (Ensembl) -> genes it binds in any of the three databases
combined_tf_to_genes = {}
for tf in causal_tfs_ensembl:
    genes = set()
    for db in (tfbsdb_tf_to_genes, tfbsdb2_tf_to_genes, chea_seq):
        if tf in db:
            genes.update(db[tf])
    combined_tf_to_genes[tf] = list(genes)

# Expression of the causal TFs (TFs x patients); TFs without expression data are dropped
tf_exp = exp_data.reindex(causal_tfs_symbols).dropna()
print(f"TFs with expression data: {tf_exp.shape[0]}")

TFs kept: 296 / 345 (>= 1.0 TPM in >= 20% of tumours)
Dropped: ['AR', 'ARX', 'BARHL1', 'BHLHE22', 'CDX1', 'CPEB1', 'DBX1', 'DRGX', 'EMX1', 'EMX2', 'ESRRG', 'FOXA2', 'FOXG1', 'GFI1B', 'HESX1', 'HMX2', 'HNF1A', 'HOXA13', 'HOXB5', 'HOXD12', 'KLF14', 'KLF15', 'LHX3', 'LHX9', 'MESP1', 'MIXL1', 'MNX1', 'MYOD1', 'NKX2-1', 'NKX2-3', 'NKX2-6', 'NR5A2', 'ONECUT1', 'ONECUT2', 'PAX1', 'PAX6', 'PGR', 'PITX3', 'PKNOX2', 'POU4F1', 'SOX11', 'SOX14', 'SOX30', 'SOX5', 'TBR1', 'TLX2', 'VAX1', 'VENTX', 'ZBTB16']
TFs from causal results: 296 | mapped to Ensembl: 296
TFs with expression data: 296


In [5]:
# TF-TF network with LASSO: A -> B if A has a binding site in B's promoter AND A's expression predicts B's (non-zero coef)
start = time.time()
tf_tf_edges = []
print(f"Building TF-TF network for {tf_exp.shape[0]} TFs...")

for i, target_tf in enumerate(tf_exp.index):
    if i % 25 == 0:
        print(f"{i}/{tf_exp.shape[0]}")

    # Each TF in turn is the target whose expression is to be predicted
    target_ensembl = symbol_to_ensembl.get(target_tf)
    if target_ensembl is None:
        continue

    # Candidate regulators: causal TFs with a binding site in the target's promoter (needs at least 2)
    predictor_tfs = []
    for source_ensembl, target_genes in combined_tf_to_genes.items():
        if target_ensembl in target_genes:
            source_symbol = ensembl_to_symbol.get(source_ensembl)
            if source_symbol and source_symbol in tf_exp.index and source_symbol != target_tf:
                predictor_tfs.append(source_symbol)
    if len(predictor_tfs) < 2:
        continue

    # LASSO: predict the target's expression from its candidate regulators
    # Non-zero coefficients become edges; the sign gives activation (1) or repression (-1)
    X_scaled = StandardScaler().fit_transform(tf_exp.loc[predictor_tfs].T.values)
    y = tf_exp.loc[target_tf].values
    try:
        lasso = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_scaled, y)
        for j, coef in enumerate(lasso.coef_):
            if coef != 0:
                tf_tf_edges.append({'Source': predictor_tfs[j], 'Target': target_tf,
                                    'Edge': 1 if coef > 0 else -1, 'Coef': coef})
    except Exception:
        continue

tf_network = pd.DataFrame(tf_tf_edges)
print(f"Completed in {(time.time()-start)/60:.1f} minutes")
print(f"TF-TF network edges: {len(tf_network)} (GBM: 11848)")
print(f"Unique source TFs: {tf_network['Source'].nunique()} | unique target TFs: {tf_network['Target'].nunique()}")
tf_network.to_csv(os.path.join(output_dir, "tf_tf_network.csv"), index=False)

Building TF-TF network for 296 TFs...
0/296
25/296
50/296
75/296
100/296
125/296
150/296
175/296
200/296
225/296
250/296
275/296
Completed in 2.7 minutes
TF-TF network edges: 14331 (GBM: 11848)
Unique source TFs: 296 | unique target TFs: 296


In [6]:
# out/in degree ratio: high = master regulator (regulates many, regulated by few)
outdegree = tf_network.groupby('Source')['Target'].count().rename('outdegree')
indegree  = tf_network.groupby('Target')['Source'].count().rename('indegree')
network_degrees = pd.DataFrame({'outdegree': outdegree, 'indegree': indegree}).fillna(0)
network_degrees['out/in'] = network_degrees['outdegree'] / (network_degrees['indegree'] + 1)
network_degrees['Alt_ID'] = network_degrees.index       # miner source expects this column
network_degrees = network_degrees.sort_values('out/in', ascending=False)
print("Top 10 TFs by outdegree/indegree ratio:")
print(network_degrees.head(10))
network_degrees.to_csv(os.path.join(output_dir, "tf_network_degrees.csv"))

# Coverage dictionary: TF -> TFs it regulates
coverage_dict = {tf: tf_network.loc[tf_network['Source'] == tf, 'Target'].tolist() for tf in tf_network['Source'].unique()}

Top 10 TFs by outdegree/indegree ratio:
        outdegree  indegree    out/in  Alt_ID
IRF9           87        21  3.954545    IRF9
SRF           135        38  3.461538     SRF
ELF4          118        34  3.371429    ELF4
POU2F3         89        27  3.178571  POU2F3
NKX3-1         65        20  3.095238  NKX3-1
TCF7L1        101        35  2.805556  TCF7L1
FOXP1         142        50  2.784314   FOXP1
TFAP2E        128        45  2.782609  TFAP2E
FOXO1         105        37  2.763158   FOXO1
SPI1          108        39  2.700000    SPI1


In [7]:
# Master regulators per state, following the paper (Methods, "Identification of master regulators from TF-TF network"):
#   0. TFs causally associated with the state: regulons active in the state (Welch t-test of eigengenes, state vs rest,
#      p <= STATE_P, and over- or under-active in >= STATE_FRAC of the state's patients); then the causal TFs whose causal regulons
#      are over-represented among them (hypergeometric p <= TF_ENRICH_P). Without the enrichment step almost every TF has at
#      least one active regulon in every state (BRCA: 110-200 of 263 per state), so all states get the same network
#   1. LASSO per TF: predict each state TF's expression from the other state TFs that have a binding site in its promoter
#      (TFBSDB v1/v2 or ChEA)
#   2. Pruning: the largest penalty whose cross-validated error is within one standard error of the best one
#      (fewest TF predictors for the same predictive accuracy: the "one-standard-error rule")
#   3. Non-zero LASSO coefficients = weighted edges predictor -> target. TFs ranked by outdegree / indegree; top N_MASTER kept
# Not specified in the paper, chosen here: STATE_FRAC, STATE_TF_RULE, LASSO_SAMPLES, the one-standard-error rule, N_MASTER
# (the paper reports 67 master regulators over its states, ~3 per state) and indegree + 1 in the ratio
# (a TF with no regulators would otherwise divide by zero)
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import KFold
from joblib import Parallel, delayed
from tqdm.auto import tqdm

N_MASTER      = 3        # master regulators kept per state
STATE_P       = 0.05     # differential regulon activity, state vs rest
STATE_FRAC    = 0.5      # regulon over- or under-active in >= this fraction of the state's patients (miner's state/program threshold)
STATE_TF_RULE = "enriched" # "enriched": TFs whose causal regulons are over-represented in the state; "any": >= 1 active regulon
TF_ENRICH_P   = 0.05     # hypergeometric p-value for "enriched"
LASSO_SAMPLES = "all"    # "all": LASSO fitted on every patient, the state sets the TF list; "state": only the state's patients
MIN_STATE_N   = 20       # with LASSO_SAMPLES = "state", smaller states are skipped (too few patients for cross-validation)
CV_FOLDS      = 5
N_JOBS        = -1

# Quantised regulon activity (+1 over, -1 under, 0 neither), regulons x patients, from 02
over_m  = pd.read_csv(os.path.join(output_dir, "overExpressedMembers.csv"), index_col=0)
under_m = pd.read_csv(os.path.join(output_dir, "underExpressedMembers.csv"), index_col=0)
reg_activity = over_m.astype(int) - under_m.astype(int)
reg_activity.index = reg_activity.index.astype(str)
del over_m, under_m

# Binding sites: target TF (symbol) -> TFs (symbols) with a binding site in its promoter, among the expressed causal TFs
tf_exp_set = set(tf_exp.index)
tf_of_ensembl = {symbol_to_ensembl[t]: t for t in tf_exp.index if t in symbol_to_ensembl}
binders_of = {}
for src_ens, target_genes in combined_tf_to_genes.items():
    src = ensembl_to_symbol.get(src_ens)
    if src not in tf_exp_set:
        continue
    for t_ens in target_genes:
        t = tf_of_ensembl.get(t_ens)
        if t is not None and t != src:
            binders_of.setdefault(t, set()).add(src)


def pruned_lasso(target, predictors, X, y, cv_folds=5, seed=42):
    '''
    LASSO of one target TF on its candidate regulators, pruned with the one-standard-error rule.

    Args: target (str), predictors (list of str). TF symbols
          X (np.array), shape (n_patients, n_predictors). Predictor expression; y (np.array), shape (n_patients,). Target expression
    Returns: target, predictors (as given), coefficients after pruning (np.array, aligned with predictors),
             number of predictors at the CV-best penalty (before pruning)
    '''
    import numpy as np
    from sklearn.linear_model import LassoCV, Lasso
    from sklearn.model_selection import KFold

    # Predictors with no variation in these patients cannot be used
    sd = X.std(0)
    ok = sd > 0
    coefs = np.zeros(len(predictors))
    if ok.sum() == 0 or y.std() == 0:
        return target, predictors, coefs, 0
    Xs = (X[:, ok] - X[:, ok].mean(0)) / sd[ok]
    ys = (y - y.mean()) / y.std()

    # Shuffled folds: patients may be ordered by platform or batch
    cv = LassoCV(cv=KFold(cv_folds, shuffle=True, random_state=seed), max_iter=10000).fit(Xs, ys)

    # One-standard-error rule: largest penalty (fewest predictors) whose mean CV error <= best mean + its standard error
    mse = cv.mse_path_.mean(axis=1)
    se  = cv.mse_path_.std(axis=1, ddof=1) / np.sqrt(cv.mse_path_.shape[1])
    best = np.argmin(mse)
    alpha_1se = cv.alphas_[mse <= mse[best] + se[best]].max()

    coefs[ok] = Lasso(alpha=alpha_1se, max_iter=10000).fit(Xs, ys).coef_
    return target, predictors, coefs, int((cv.coef_ != 0).sum())


# Causal regulons of each TF (from the driver-filtered causal flows), for the enrichment test
regulons_of_tf = causal_results.groupby("Regulator")["Regulon"].apply(lambda r: set(r.astype(str))).to_dict()
all_causal_regulons = set(causal_results["Regulon"].astype(str))

lasso_cache = {}              # (target, predictors, sample set) -> pruned coefficients; identical models are fitted once
state_master_regulators_all = {}
mr_rows, edge_rows, state_rows = [], [], []
start = time.time()

for state_id, state_patients in tqdm(transcriptional_states.items(), desc="states"):
    state_pts = [p for p in state_patients if p in eigengenes.columns]
    state_pt_set = set(state_pts)
    rest_pts  = [p for p in eigengenes.columns if p not in state_pt_set]
    if len(state_pts) < 3 or len(rest_pts) < 3:
        continue

    # 0. Regulons active in this state: differential eigengene (Welch t-test) and active in >= STATE_FRAC of its patients
    _, pvals = scipy_stats.ttest_ind(eigengenes[state_pts].values, eigengenes[rest_pts].values, axis=1, equal_var=False)
    diff_regs = set(eigengenes.index.astype(str)[pvals <= STATE_P])
    frac = reg_activity.reindex(columns=[p for p in state_pts if p in reg_activity.columns]).mean(axis=1)
    consistent_regs = set(frac.index[frac.abs() >= STATE_FRAC])
    state_regulons = diff_regs & consistent_regs

    # Causal TFs of those regulons (expressed, with expression data)
    if STATE_TF_RULE == "enriched":
        # Hypergeometric test per TF: of all causal regulons, are this TF's ones over-represented among the state's active ones?
        state_causal = state_regulons & all_causal_regulons
        state_tfs = []
        for tf, tf_regs in regulons_of_tf.items():
            k = len(tf_regs & state_causal)
            if tf in tf_exp_set and k > 0 and \
               scipy_stats.hypergeom.sf(k - 1, len(all_causal_regulons), len(tf_regs), len(state_causal)) <= TF_ENRICH_P:
                state_tfs.append(tf)
        state_tfs = sorted(state_tfs)
    else:
        state_tfs = sorted(set(causal_results.loc[causal_results["Regulon"].isin(state_regulons), "Regulator"]) & tf_exp_set)

    # Patients the LASSO is fitted on
    if LASSO_SAMPLES == "state":
        samples = [p for p in state_patients if p in tf_exp.columns]
        if len(samples) < MIN_STATE_N:
            print(f"State {state_id}: {len(samples)} patients < {MIN_STATE_N}, skipped")
            continue
        sample_key = f"state_{state_id}"
    else:
        samples, sample_key = list(tf_exp.columns), "all"

    # 1. One LASSO per state TF, predictors = state TFs with a binding site in its promoter
    state_set = set(state_tfs)
    tasks = []
    for target in state_tfs:
        preds = tuple(sorted(binders_of.get(target, set()) & state_set))
        if preds:
            tasks.append((target, preds, (target, preds, sample_key)))
    todo = [t for t in tasks if t[2] not in lasso_cache]
    X_all = tf_exp.loc[:, samples]
    results = Parallel(n_jobs=N_JOBS)(
        delayed(pruned_lasso)(target, preds, X_all.loc[list(preds)].T.values, X_all.loc[target].values, CV_FOLDS)
        for target, preds, _ in todo)
    for (target, preds, key), res in zip(todo, results):
        lasso_cache[key] = res

    # 2-3. Edges = non-zero pruned coefficients
    edges = []
    n_before = 0
    for target, preds, key in tasks:
        _, _, coefs, n_cv_best = lasso_cache[key]
        n_before += n_cv_best
        for src, c in zip(preds, coefs):
            if c != 0:
                edges.append({"State": state_id, "Source": src, "Target": target, "Coef": float(c)})
    edge_rows.extend(edges)
    state_rows.append({"State": state_id, "n_patients": len(state_pts), "n_state_regulons": len(state_regulons),
                       "n_state_tfs": len(state_tfs), "n_lasso_models": len(tasks),
                       "edges_before_pruning": n_before, "edges_after_pruning": len(edges)})
    if not edges:
        continue

    # Rank by outdegree / (indegree + 1); ties broken by outdegree, then by the summed |coefficient| of outgoing edges
    e = pd.DataFrame(edges)
    deg = pd.DataFrame({"outdegree": e.groupby("Source").size(), "indegree": e.groupby("Target").size(),
                        "out_weight": e.assign(w=e["Coef"].abs()).groupby("Source")["w"].sum()}).fillna(0)
    deg["out/in"] = deg["outdegree"] / (deg["indegree"] + 1)
    deg = deg[deg["outdegree"] > 0].sort_values(["out/in", "outdegree", "out_weight"], ascending=False)
    top = deg.head(N_MASTER)

    # Direction: is the master regulator's own expression higher or lower in the state than in the rest?
    t_expr, p_expr = scipy_stats.ttest_ind(tf_exp.loc[top.index, [p for p in state_pts if p in tf_exp.columns]].values,
                                           tf_exp.loc[top.index, [p for p in rest_pts if p in tf_exp.columns]].values,
                                           axis=1, equal_var=False)
    state_master_regulators_all[state_id] = top.index.tolist()
    for rank, (tf, row) in enumerate(top.iterrows(), start=1):
        mr_rows.append({"State": state_id, "MasterRegulator": tf, "Rank": rank, "outdegree": int(row["outdegree"]),
                        "indegree": int(row["indegree"]), "out/in": row["out/in"], "out_weight": row["out_weight"],
                        "expr_t_state_vs_rest": t_expr[rank - 1], "expr_p_state_vs_rest": p_expr[rank - 1]})

print(f"Completed in {(time.time() - start) / 60:.1f} min | LASSO models fitted: {len(lasso_cache)}")
state_summary = pd.DataFrame(state_rows).set_index("State")
print(state_summary.to_string())
pd.DataFrame(edge_rows).to_csv(os.path.join(output_dir, "tf_tf_network_per_state.csv"), index=False)
state_summary.to_csv(os.path.join(output_dir, "master_regulator_state_summary.csv"))

# Summary across states
all_masters = sorted(set(tf for tfs in state_master_regulators_all.values() for tf in tfs))
n_states_of = pd.Series([tf for tfs in state_master_regulators_all.values() for tf in tfs]).value_counts()
print(f"\nStates with master regulators: {len(state_master_regulators_all)} / {len(transcriptional_states)}")
print(f"Unique master regulators: {len(all_masters)} (paper, GBM: 67; most state-specific)")
print(f"Master regulators of one state only: {int((n_states_of == 1).sum())} | shared by 2+ states: {int((n_states_of > 1).sum())}")
print("Most shared:", n_states_of.head(10).to_dict())

states:   0%|          | 0/21 [00:00<?, ?it/s]

Completed in 0.3 min | LASSO models fitted: 279
       n_patients  n_state_regulons  n_state_tfs  n_lasso_models  edges_before_pruning  edges_after_pruning
State                                                                                                      
0              61              1957           16              16                   108                   64
1              58              1864           13              13                    87                   51
2              26              1715           12              12                    57                   29
3              46              1431           22              22                   222                  131
4              34              1387           16              16                   138                   83
5              51              2136           22              22                   193                   96
6              37              1290           20              20                   142  

In [8]:
# Save master regulators as a long table: one row per (state, TF), read by notebook 07 (columns State, MasterRegulator)
# Also: rank within the state, network degrees, and whether the TF's own expression is higher (t > 0) or lower in the state
master_reg_df = pd.DataFrame(mr_rows, columns=["State", "MasterRegulator", "Rank", "outdegree", "indegree", "out/in",
                                               "out_weight", "expr_t_state_vs_rest", "expr_p_state_vs_rest"])
master_reg_df.to_csv(os.path.join(output_dir, "master_regulators_per_state.csv"), index=False)
print(f"Saved {len(master_reg_df)} master regulator-state associations")
print(f"Unique master regulators: {master_reg_df['MasterRegulator'].nunique()} | states covered: {master_reg_df['State'].nunique()}")

Saved 57 master regulator-state associations
Unique master regulators: 31 | states covered: 19
